In [ ]:
import os, sys
REPO_DIR = '/content/Filtre-Voix-DL'
assert os.path.exists(REPO_DIR), (
    f'{REPO_DIR} introuvable — exécute la cellule clone de main.ipynb '
    'pour cloner/mettre à jour le repo sur ce runtime Colab.'
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception as e:
    pass  # bug connu Colab Python 3.12, sans impact

print(f'sys.path OK — repo : {REPO_DIR}')

In [ ]:
import librosa
import torch
import torchaudio.transforms as T
import numpy as np
from src import config 
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import butter, filtfilt

In [ ]:
MAX_PAIRS = 3
SAMPLE_RATE = 16000

train_dir = Path(config.DATA_NOISY)
test_dir = Path(config.DATA_CLEAN) 

## FILTER

In [ ]:
def bandpass_filter(audio, sr, lowcut=80, highcut=6000, order=5):

    nyquist = sr / 2

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, audio)

## NORMALIZE

In [ ]:
def normalize_audio(audio):

    max_val = np.max(np.abs(audio))

    if max_val > 0:
        audio = audio / max_val

    return audio

In [ ]:
mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,
    hop_length=256,
    n_mels=80
)
to_db = T.AmplitudeToDB()

def make_mel(audio):

    audio_tensor = torch.from_numpy(audio.copy()).float().unsqueeze(0)

    mel = mel_transform(audio_tensor)

    mel_db = to_db(mel)

    return mel_db[0].numpy()

In [ ]:
train_files = sorted(train_dir.glob("train_*.wav"))
test_files = sorted(test_dir.glob("test_*.wav"))

print("Train founds :", len(train_files))
print("Test founds :", len(test_files))

In [ ]:
for i in range(min(MAX_PAIRS, len(train_files), len(test_files))):

    train_file = train_files[i]
    test_file = test_files[i]

    # =========================
    # LOAD AUDIO
    # =========================

    train_wave, _ = librosa.load(
        train_file,
        sr=SAMPLE_RATE,
        mono=True
    )

    test_wave, _ = librosa.load(
        test_file,
        sr=SAMPLE_RATE,
        mono=True
    )

    # =========================
    # SAME LENGTH
    # =========================

    min_length = min(
        len(train_wave),
        len(test_wave)
    )

    train_wave = train_wave[:min_length]
    test_wave = test_wave[:min_length]

    # =========================
    # FILTER
    # =========================

    train_filtered = bandpass_filter(
        train_wave,
        SAMPLE_RATE
    )

    test_filtered = bandpass_filter(
        test_wave,
        SAMPLE_RATE
    )

    # =========================
    # NORMALIZATION
    # =========================

    train_filtered = normalize_audio(
        train_filtered
    )

    test_filtered = normalize_audio(
        test_filtered
    )

    # =========================
    # MELS
    # =========================

    train_mel_filtered = make_mel(
        train_filtered
    )

    test_mel_filtered = make_mel(
        test_filtered
    )

    # =========================
    # DISPLAY
    # =========================

    fig, axs = plt.subplots(
        2,
        1,
        figsize=(12, 6)
    )

    axs[0].imshow(
        train_mel_filtered,
        origin="lower",
        aspect="auto"
    )

    axs[0].set_title(
        f"TRAIN filtered : {train_file.name}"
    )

    axs[1].imshow(
        test_mel_filtered,
        origin="lower",
        aspect="auto"
    )

    axs[1].set_title(
        f"TEST filtered : {test_file.name}"
    )

    plt.tight_layout()

    plt.show(block=True)